In [2]:
import polars as pl


In [14]:
COLUMNS_TO_KEEP = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "trip_distance",
    "passenger_count",
    "fare_amount",
    "total_amount",
]


In [34]:


df = pl.read_parquet(
    "../data/raw/yellow_tripdata_2025-01.parquet",columns=COLUMNS_TO_KEEP
)
df.describe()

statistic,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,trip_distance,passenger_count,fare_amount,total_amount
str,str,str,f64,f64,f64,f64,f64
"""count""","""3475226""","""3475226""",3.475226e6,3.475226e6,2.935077e6,3.475226e6,3.475226e6
"""null_count""","""0""","""0""",0.0,0.0,540149.0,0.0,0.0
"""mean""","""2025-01-17 11:02:55.910964""","""2025-01-17 11:17:56.997901""",165.191576,5.855126,1.297859,17.081803,25.611292
"""std""",null,null,64.529483,564.6016,0.75075,463.472918,463.658478
"""min""","""2024-12-31 20:47:55""","""2024-12-18 07:52:40""",1.0,0.0,0.0,-900.0,-901.0
"""25%""","""2025-01-10 07:59:01""","""2025-01-10 08:15:29""",132.0,0.98,1.0,8.6,15.2
"""50%""","""2025-01-17 15:41:34""","""2025-01-17 15:59:34""",162.0,1.67,1.0,12.11,19.95
"""75%""","""2025-01-24 19:34:06""","""2025-01-24 19:48:31""",234.0,3.1,1.0,19.5,27.78
"""max""","""2025-02-01 00:00:44""","""2025-02-01 23:44:11""",265.0,276423.57,9.0,863372.12,863380.37


In [35]:
df.shape

(3475226, 7)

In [36]:
df=df.rename({"tpep_pickup_datetime":"pickup_dt","tpep_dropoff_datetime":"dropoff_dt"})

In [37]:
df.head()

pickup_dt,dropoff_dt,PULocationID,trip_distance,passenger_count,fare_amount,total_amount
datetime[μs],datetime[μs],i32,f64,i64,f64,f64
2025-01-01 00:18:38,2025-01-01 00:26:59,229,1.6,1,10.0,18.0
2025-01-01 00:32:40,2025-01-01 00:35:13,236,0.5,1,5.1,12.12
2025-01-01 00:44:04,2025-01-01 00:46:01,141,0.6,1,5.1,12.1
2025-01-01 00:14:27,2025-01-01 00:20:01,244,0.52,3,7.2,9.7
2025-01-01 00:21:34,2025-01-01 00:25:06,244,0.66,3,5.8,8.3


In [38]:
df.describe()

statistic,pickup_dt,dropoff_dt,PULocationID,trip_distance,passenger_count,fare_amount,total_amount
str,str,str,f64,f64,f64,f64,f64
"""count""","""3475226""","""3475226""",3.475226e6,3.475226e6,2.935077e6,3.475226e6,3.475226e6
"""null_count""","""0""","""0""",0.0,0.0,540149.0,0.0,0.0
"""mean""","""2025-01-17 11:02:55.910964""","""2025-01-17 11:17:56.997901""",165.191576,5.855126,1.297859,17.081803,25.611292
"""std""",null,null,64.529483,564.6016,0.75075,463.472918,463.658478
"""min""","""2024-12-31 20:47:55""","""2024-12-18 07:52:40""",1.0,0.0,0.0,-900.0,-901.0
"""25%""","""2025-01-10 07:59:01""","""2025-01-10 08:15:29""",132.0,0.98,1.0,8.6,15.2
"""50%""","""2025-01-17 15:41:34""","""2025-01-17 15:59:34""",162.0,1.67,1.0,12.11,19.95
"""75%""","""2025-01-24 19:34:06""","""2025-01-24 19:48:31""",234.0,3.1,1.0,19.5,27.78
"""max""","""2025-02-01 00:00:44""","""2025-02-01 23:44:11""",265.0,276423.57,9.0,863372.12,863380.37


In [39]:
df=df.drop_nulls(subset=['pickup_dt'])

In [40]:
df.shape

(3475226, 7)

In [41]:
df=df.filter(
    (pl.col("trip_distance")>0) &
    (pl.col("fare_amount")>0 ) &
    (pl.col("PULocationID").is_between(1,263)) &
    (pl.col("pickup_dt").dt.year()==2025)
)

In [42]:
df.shape

(3245616, 7)

In [43]:
df=df.with_columns(
    pl.col("pickup_dt")
    .dt.truncate("1h")
    .alias("pickup_hour")
)

In [44]:
agg_df=df.group_by(["pickup_hour","PULocationID"]).len().rename({"len":"trip_count"})

In [45]:
agg_df[13500:135510]

pickup_hour,PULocationID,trip_count
datetime[μs],i32,u32
2025-01-14 22:00:00,256,2
2025-01-29 17:00:00,10,1
2025-01-19 13:00:00,186,277
2025-01-29 12:00:00,54,1
2025-01-05 06:00:00,170,8
…,…,…
2025-01-31 07:00:00,232,10
2025-01-06 14:00:00,53,1
2025-01-28 05:00:00,20,2


In [63]:
agg_df.sort(['pickup_hour']).tail()

pickup_hour,PULocationID,trip_count
datetime[μs],i32,u32
2025-01-31 23:00:00,65,15
2025-01-31 23:00:00,45,43
2025-01-31 23:00:00,186,113
2025-01-31 23:00:00,215,1
2025-02-01 00:00:00,113,1


In [46]:
agg_df.shape

(91095, 3)

In [47]:
all_hours=pl.datetime_range(
    start=df['pickup_dt'].min().replace(minute=0,second=0,microsecond=0),
    end=df['pickup_dt'].max().replace(minute=0,second=0,microsecond=0),
    interval='1h',
    eager=True
).alias("pickup_hour")


In [48]:
all_hours

pickup_hour
datetime[μs]
2025-01-01 00:00:00
2025-01-01 01:00:00
2025-01-01 02:00:00
2025-01-01 03:00:00
2025-01-01 04:00:00
…
2025-01-31 20:00:00
2025-01-31 21:00:00
2025-01-31 22:00:00


In [49]:
all_zones=pl.Series("PULocationID",list(range(1,264)))


In [50]:
all_zones

PULocationID
i64
1
2
3
4
5
…
259
260
261


In [52]:
type(all_hours)

polars.series.series.Series

In [57]:
all_hours.shape

(745,)

In [53]:
type(all_hours.to_frame())

polars.dataframe.frame.DataFrame

In [55]:
full_grid=all_hours.to_frame().join(all_zones.to_frame(),how='cross')

In [56]:
full_grid.shape

(195935, 2)

In [59]:
result=(
    full_grid
    .join(agg_df,on=['pickup_hour',"PULocationID"],how='left')
    .with_columns(pl.col("trip_count").fill_null(0))
    .sort(['pickup_hour',"PULocationID"])
)

In [8]:
train_df=pl.read_parquet("../data/processed/train.parquet")
stream_df=pl.read_parquet("../data/processed/stream.parquet")

In [9]:
train_df.head()

pickup_hour,PULocationID,trip_count
datetime[μs],i32,u32
2025-01-01 00:00:00,4,26
2025-01-01 00:00:00,7,6
2025-01-01 00:00:00,9,1
2025-01-01 00:00:00,10,1
2025-01-01 00:00:00,12,1


In [10]:
train_df.tail()

pickup_hour,PULocationID,trip_count
datetime[μs],i32,u32
2025-06-30 23:00:00,257,2
2025-06-30 23:00:00,258,2
2025-06-30 23:00:00,261,10
2025-06-30 23:00:00,262,15
2025-06-30 23:00:00,263,32


In [11]:
stream_df.head()

pickup_hour,PULocationID,trip_count
datetime[μs],i32,u32
2025-07-01 00:00:00,3,1
2025-07-01 00:00:00,4,9
2025-07-01 00:00:00,7,9
2025-07-01 00:00:00,10,5
2025-07-01 00:00:00,13,12


In [12]:
stream_df.tail()

pickup_hour,PULocationID,trip_count
datetime[μs],i32,u32
2025-12-31 23:00:00,259,1
2025-12-31 23:00:00,260,6
2025-12-31 23:00:00,261,24
2025-12-31 23:00:00,262,68
2025-12-31 23:00:00,263,153


In [60]:
result.shape

(195935, 3)